In [4]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings

# Suppress warnings
warnings.filterwarnings("ignore")

# --- Configuration ---
INPUT_FILE = 'features.csv'
OUTPUT_FILE = 'xgboost_results.csv'

def mean_absolute_percentage_error(y_true, y_pred): 
    """Calculates MAPE (Mean Absolute Percentage Error)"""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def main():
    print("Loading data...")
    try:
        df = pd.read_csv(INPUT_FILE)
    except FileNotFoundError:
        print(f"Error: {INPUT_FILE} not found.")
        return

    # 1. Data Prep
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values(['Store', 'Date'])

    # 2. Feature Selection
    # Exclude ID cols and Target cols
    exclude_cols = ['Store', 'Date', 'Weekly_Sales', 'sales_log1p']
    
    # Select all other numeric columns
    feature_cols = [c for c in df.columns if c not in exclude_cols]
    feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df[c])]
    
    print(f"Selected {len(feature_cols)} features for XGBoost.")

    stores = df['Store'].unique()
    results = []
    
    print(f"Training XGBoost models for {len(stores)} stores...")

    for store in stores:
        store_df = df[df['Store'] == store].copy()
        
        # 3. Train/Test Split (80% Train, 20% Test - Chronological)
        n_rows = len(store_df)
        train_size = int(n_rows * 0.8)
        
        train = store_df.iloc[:train_size]
        test = store_df.iloc[train_size:]
        
        if len(test) == 0:
            continue

        # 4. Prepare Data
        X_train = train[feature_cols]
        y_train = train['sales_log1p'] # Train on Log Sales
        
        X_test = test[feature_cols]
        y_test_real = test['Weekly_Sales'] # Evaluate on Real Sales
        
        try:
            # 5. Initialize and Train XGBoost
            model = xgb.XGBRegressor(
                objective='reg:squarederror',
                n_estimators=1000,
                learning_rate=0.05,
                max_depth=6,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                n_jobs=-1  # Use all cores
            )
            
            model.fit(X_train, y_train)
            
            # 6. Forecast
            preds_log = model.predict(X_test)
            
            # Inverse Transform (Log -> Real Dollars)
            preds_real = np.expm1(preds_log)
            
            # 7. Metrics
            mae = mean_absolute_error(y_test_real, preds_real)
            rmse = np.sqrt(mean_squared_error(y_test_real, preds_real))
            mape = mean_absolute_percentage_error(y_test_real, preds_real)
            
            results.append({
                'Store ID': store,
                'MAE': round(mae, 2),
                'RMSE': round(rmse, 2),
                'MAPE (%)': round(mape, 2)
            })
            
        except Exception as e:
            print(f"Store {store} Failed: {e}")
            results.append({
                'Store ID': store,
                'MAE': np.nan,
                'RMSE': np.nan,
                'MAPE (%)': np.nan
            })

    # 8. Results Table
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values(by='MAPE (%)', ascending=True)
    
    print("\n--- XGBoost Final Results (Sorted by MAPE) ---")
    print(results_df.to_string(index=False))
    
    results_df.to_csv(OUTPUT_FILE, index=False)
    print(f"\nResults saved to {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

Loading data...
Selected 34 features for XGBoost.
Training XGBoost models for 45 stores...

--- XGBoost Final Results (Sorted by MAPE) ---
 Store ID       MAE      RMSE  MAPE (%)
       43  13899.54  16648.73      2.24
       37  13037.03  16574.17      2.48
       34  27712.60  36236.68      2.81
       32  33413.35  42893.61      2.86
       22  29223.04  39269.95      3.00
        2  56527.16  71831.48      3.01
       42  17628.19  21977.20      3.07
       30  14183.15  17149.10      3.22
       31  44404.10  52500.58      3.24
       10  59702.39  73376.76      3.38
        8  30713.58  38836.54      3.41
        6  53601.55  73247.18      3.46
       24  48435.50  58525.19      3.49
       25  26079.66  31786.61      3.65
       26  38930.38  49340.73      3.73
       12  36517.64  44246.38      3.75
       17  37071.78  47963.89      3.81
       45  29480.17  34185.67      3.94
       41  55160.58  74312.48      3.95
       18  41733.97  52322.11      4.03
        9  22728.37  